In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import scipy.sparse
import matplotlib.pyplot as plt
import torch.nn.functional as F


In [56]:
# Set the data to be loaded
# This raises a pytorch error but I think it's ok?
X = torch.load('./data/midwest/X-midwest2-100d.pt')
y = torch.load('./data/midwest/y-midwest2-100d.pt') 

/tmp/ipykernel_637823/2817959694.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X = torch.load('./data/midwest/X-midwest2-100d.pt')
/tmp/ipykernel_637823/2817959694.py:

In [57]:
# Train test split
# Use stratisfy!
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2)

In [60]:
import pickle

def load_dictionary_pickle(filename):
    """Loads a dictionary from a pickle file."""
    with open(filename, 'rb') as f:  # 'rb' for binary read
        return pickle.load(f)

# import dictionary and set vocab size
embedding_matrix = load_dictionary_pickle('./data/midwest/em-midwest-100d.pkl')
vocab_size, embedding_dim = list(embedding_matrix.size())

In [61]:
# Check
print('vocab size', vocab_size)
print('embedding dimension', embedding_dim)

vocab size 20001
embedding dimension 100


Define the LSTM model here
I've tried different hidden sizes and whatnot, they don't seem to make a huge difference


In [62]:
#Define Model (this is taken from Gemini--can it be improved?)
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, output_size, embedding_matrix):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(embedding_matrix) #load the pretrained embeddings.
        
        # Freeze embeddings. Tried both ways, "false" freezes the embeddings
        self.embedding.weight.requires_grad = False 

        # Tried inserting dropout. Not sure
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, dropout=0.3, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.lstm(embedded)
        out = self.fc(out[:, -1, :])
        return out


In [63]:
# Use the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)


cuda


In [64]:
# Set model parameters
hidden_size = 196 # Tried larger size here, not sure
num_layers = 2
output_size = 3  # Number of output classes: positive/negative/neutral

## Idea is to add an attention layer
model = LSTMModel(vocab_size, embedding_dim, hidden_size, num_layers, output_size, embedding_matrix)
model.to(device)

LSTMModel(
  (embedding): Embedding(20001, 100)
  (lstm): LSTM(100, 196, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=196, out_features=3, bias=True)
)

In [65]:
from torchinfo import summary
summary(model)

# Consier freezing embedding parameters

Layer (type:depth-idx)                   Param #
LSTMModel                                --
├─Embedding: 1-1                         (2,000,100)
├─LSTM: 1-2                              542,528
├─Linear: 1-3                            591
Total params: 2,543,219
Trainable params: 543,119
Non-trainable params: 2,000,100

In [66]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

class SentimentDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences 
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [67]:
from sklearn.utils import class_weight
# Since the data classes are biased towards positive, this may be a way to improve accuracy
# by weighting the cross entropy loss function

def calculate_class_weights(labels):
    """Calculates class weights for imbalanced datasets."""
    class_weights = class_weight.compute_class_weight(
        'balanced',
        classes=np.unique(labels),
        y=labels
    )
    return torch.tensor(class_weights, dtype=torch.float)


In [73]:
# Load the datasets and dataloaders
dataset_train = SentimentDataset(X_train, y_train)
dataset_test = SentimentDataset(X_test, y_test)

# Modify these
batch_size = 128
learning_rate = .001
weight_decay = .01

dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=True)

# Define loss function and optimizer. Calculate class weights of the train set first
calculate_class_weights(y_train.tolist())
criterion = nn.CrossEntropyLoss(weight=class_weights) 

# Next step could be to use a learning rate scheduler
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [69]:
def accuracy_fn(preds, tru):
    preds_labels = torch.argmax(preds, dim=1)
    return int(torch.eq(preds_labels, tru).sum())

This will automatically generate a plot after the training/testing cycle is complete

In [70]:
import datetime

def plot_lists_with_colors(list1, list2, label1="List 1", label2="List 2", color1="blue", color2="red"):
    # Ensure lists have equal length (or adjust as needed)
    min_len = min(len(list1), len(list2))
    x_values = np.arange(min_len)

    # Create the plot
    plt.figure(figsize=(10, 6))  # Adjust figure size if needed

    # Plot List 1
    plt.plot(x_values, list1[:min_len], color=color1, label=label1, linestyle='none', marker='o')

    # Plot List 2
    plt.plot(x_values, list2[:min_len], color=color2, label=label2, linestyle='none', marker='o')

    current_time = datetime.datetime.now()
    
    # Add Labels and Title
    plt.xlabel("Epoch") 
    plt.ylabel("Accuracy (%)")
    plt.title(f"LSTM - Midwest data ({current_time.strftime("%Y-%m-%d %H:%m")})")

    # Add Legend
    plt.legend()

    # Add Grid (Optional)
    plt.grid(True)

    
    file_name = f'./sshots/model-{current_time.strftime("%Y-%m-%d--%H%m")}.png'
    plt.savefig(file_name)
    
    # Show the Plot
    plt.show()


Main train/test loop

In [ ]:
from tqdm import tqdm
import time

# Train test loops
train_accs = []
test_accs = []

epochs = 10
for epoch in range(epochs):
    # Training cycle
    model.train() # Set model to training mode
    
    train_running_loss = 0.0
    train_acc = 0
    
    for X_batch, y_batch in tqdm(dataloader_train, desc=f"Epoch {epoch + 1}/{epochs}"):
        
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        preds = model(X_batch)
        
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item()
        train_acc += accuracy_fn(preds, y_batch)

    train_acc = 100 * train_acc / (batch_size * len(dataloader_train))
    train_accs.append(train_acc)
    train_loss = train_running_loss / len(dataloader_train)

    # Testing loop
    print('Testing...', end='\r')
    model.eval() # Set model to evaluation mode
    test_running_loss = 0.0
    test_acc = 0

    with torch.no_grad():  # Disable gradient calculations during testing
        for X_batch_test, y_batch_test in dataloader_test:
            
            X_batch_test = X_batch_test.to(device)
            y_batch_test = y_batch_test.to(device)

            test_preds = model(X_batch_test)
            test_loss = criterion(test_preds, y_batch_test)

            test_running_loss += loss.item()
            test_acc += accuracy_fn(test_preds, y_batch_test)

    test_acc = 100 * test_acc / (batch_size * len(dataloader_test))
    test_accs.append(test_acc)
    test_loss = test_running_loss / len(dataloader_test)

    print(f'Train loss {train_loss:.5f}, acc {train_acc:.2f}% | Test loss {test_loss:.5f}, acc {test_acc:.2f}%')
    
plot_lists_with_colors(train_accs, test_accs, label1="Train accuracy", label2="Test accuracy")

Epoch 1/10: 100%|█████████████████████████████████████| 78125/78125 [03:59<00:00, 326.51it/s]


Train loss 0.44489, acc 73.0873 | Test loss 0.47543, acc 73.7748


Epoch 2/10: 100%|█████████████████████████████████████| 78125/78125 [04:03<00:00, 321.20it/s]


Train loss 0.42980, acc 74.2169 | Test loss 0.45144, acc 74.7115


Epoch 3/10: 100%|█████████████████████████████████████| 78125/78125 [04:09<00:00, 312.89it/s]


...

Epoch 4/10: 100%|█████████████████████████████████████| 78125/78125 [04:15<00:00, 306.03it/s]


Train loss 0.42330, acc 74.6793 | Test loss 0.39192, acc 74.5064


Epoch 5/10: 100%|█████████████████████████████████████| 78125/78125 [04:10<00:00, 311.41it/s]


Train loss 0.42167, acc 74.7725 | Test loss 0.37094, acc 74.3844


Epoch 6/10:  76%|████████████████████████████▏        | 59405/78125 [03:05<00:57, 326.66it/s]

In [35]:
## Save the model here
torch.save(model, f'./saved-models/model-midwest-weighted-cross-entropy.pth')